# Phase 5b — Active Learning Priority Queue (Move 2)

**Plan ref**: §6 Stage 5 + §2.5 Move 2

This notebook demonstrates the canonical priority formula:

```
score = α·uncertainty + β·downstream_impact_norm + γ·tier_weight
```

where:
- `uncertainty  = (1 - fusion_agreement_rate) + λ·llm_coherence_variance`
- `downstream_impact` = linked chunk count + linked keyword count
- `tier_weight(primary) = 1.0`, `tier_weight(secondary) = 0.3`
- defaults: α=0.40, β=0.40, γ=0.20, λ=0.30

Sections:
1. Pre-flight checks
2. Generate priority queue from NEEDS_REVIEW pages
3. Score distribution analysis
4. Persist priority scores back to Neo4j
5. Weight sensitivity analysis (ablation)
6. Write artifact

In [ ]:
import sys, json, logging
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv()

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')
logging.getLogger('neo4j.notifications').setLevel(logging.WARNING)

ARTIFACT_DIR = Path('_artifacts/05b_active_learning')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete')

In [ ]:
from apps.backend.graph.neo4j_client import get_driver

driver = get_driver()
print('Neo4j connected')

## Pre-flight

In [ ]:
with driver.session() as s:
    nr = s.run(
        "MATCH (p:PAGE) WHERE p.evaluationDecision IN ['needs_review','failed'] RETURN count(p) AS n"
    ).single()['n']
    scored = s.run(
        "MATCH (p:PAGE) WHERE p.priorityScore IS NOT NULL RETURN count(p) AS n"
    ).single()['n']

print(f'NEEDS_REVIEW + FAILED pages: {nr:,}')
print(f'Already scored:              {scored:,}')

## Generate Priority Queue

In [ ]:
from apps.backend.feedback.active_learning import (
    PriorityWeights,
    get_review_queue,
    compute_priority_score,
)

weights = PriorityWeights()  # default: α=0.40 β=0.40 γ=0.20
queue = get_review_queue(driver, max_pages=200, weights=weights)

print(f'Queue size: {len(queue)}')
if queue:
    print('\nTop 5 pages by priority:')
    for item in queue[:5]:
        print(f'  score={item.priority_score:.4f} tier={item.tier} cer={item.inter_engine_cer:.3f} '
              f'far={item.fusion_agreement_rate:.3f} impact={item.downstream_impact} '
              f'cls={item.problem_class}')
    print('\nBottom 5 pages by priority:')
    for item in queue[-5:]:
        print(f'  score={item.priority_score:.4f} tier={item.tier} cer={item.inter_engine_cer:.3f}')

## Score Distribution

In [ ]:
if queue:
    scores = [item.priority_score for item in queue]
    primary_scores = [item.priority_score for item in queue if item.tier == 'primary']
    secondary_scores = [item.priority_score for item in queue if item.tier == 'secondary']

    import statistics
    print('Score distribution (all):')
    print(f'  min={min(scores):.4f}  max={max(scores):.4f}  mean={statistics.mean(scores):.4f}')
    if len(scores) > 1:
        print(f'  stdev={statistics.stdev(scores):.4f}')

    print(f'\nPrimary tier: {len(primary_scores)} pages')
    if primary_scores:
        print(f'  mean={statistics.mean(primary_scores):.4f}')

    print(f'Secondary tier: {len(secondary_scores)} pages')
    if secondary_scores:
        print(f'  mean={statistics.mean(secondary_scores):.4f}')

    # Score distribution by problem class
    from collections import defaultdict
    by_class: dict = defaultdict(list)
    for item in queue:
        by_class[item.problem_class].append(item.priority_score)
    print('\nMean score by problem class:')
    for cls, cls_scores in sorted(by_class.items()):
        print(f'  {cls:30s}: {statistics.mean(cls_scores):.4f} (n={len(cls_scores)})')

## Persist Priority Scores to Neo4j

In [ ]:
from apps.backend.feedback.active_learning import update_priority_scores

n_updated = update_priority_scores(driver, weights=weights)
print(f'Updated priorityScore on {n_updated:,} PAGE nodes')

## Weight Sensitivity Ablation

How does the queue ordering change under alternative weight configurations?

In [ ]:
if queue:
    ablation_configs = [
        ('default',         PriorityWeights()),
        ('uncertainty-heavy',  PriorityWeights(alpha=0.70, beta=0.20, gamma=0.10)),
        ('impact-heavy',    PriorityWeights(alpha=0.20, beta=0.70, gamma=0.10)),
        ('tier-heavy',      PriorityWeights(alpha=0.20, beta=0.20, gamma=0.60)),
    ]

    # For each config, show the top-3 page IDs
    for name, w in ablation_configs:
        # Re-score using the raw page dicts from the first queue call
        rescored = []
        impacts = [item.downstream_impact for item in queue]
        max_impact = max(impacts) if impacts else 1
        for item in queue:
            raw = {
                'fusion_agreement_rate': item.fusion_agreement_rate,
                'downstream_impact': item.downstream_impact,
                'tier': item.tier,
            }
            rescored.append((item.page_id[:40], compute_priority_score(raw, w, max_downstream_impact=max_impact)))
        rescored.sort(key=lambda x: x[1], reverse=True)
        top3 = [f'{pid} ({s:.3f})' for pid, s in rescored[:3]]
        print(f'{name:22s}: top3 = {top3}')

## Write Artifact

In [ ]:
from datetime import datetime, timezone

queue_sample = [item.to_dict() for item in queue[:20]]

report = {
    'phase': '05b_active_learning',
    'ts': datetime.now(timezone.utc).isoformat(),
    'queue_size': len(queue),
    'pages_scored': n_updated,
    'weights': {
        'alpha': weights.alpha,
        'beta': weights.beta,
        'gamma': weights.gamma,
        'lambda': weights.lambda_,
    },
    'top_20_queue': queue_sample,
}

out = ARTIFACT_DIR / 'report.json'
out.write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(f'Artifact written → {out}')
print(f'Queue size: {len(queue)}, pages scored: {n_updated}')